# HADES —  test

End-to-end check that the repo works: score with the published model, train a small
model on the bundled sample data, then score with that new model.


## 0. Imports

All imports live here so nothing later depends on cell order.

In [1]:
from pathlib import Path

from HADES_CALCULATOR import HADES_score
from train_your_HADES import HADESTrainer
from parameters_test import custom_param_spaces as small_param_spaces

DATA = "data_to_test.csv"
assert Path(DATA).exists(), f"Run this notebook from the repo root; {DATA} not found."
print("imports ok")

imports ok


## 1. Score with the published model

`HADES_score()` with no arguments loads `HADES_saved_models/` and derives the descriptor
columns from `train_test_saved/X_train.parquet`.

In [4]:
hades = HADES_score()
single = hades.score("CCCC")
print(f" HADES_score(butane) = {single:.6f}")

 HADES_score(butane) = 0.000458


In [5]:
batch = hades.score(["CCCC", "CCO", "c1ccccc1"])
print(f"batch scores: {batch.shape}")
batch

batch scores: (3, 2)


,SMILES,HADES_Score
0,CCCC,0.000458
1,CCO,0.017847
2,c1ccccc1,0.011470


Whole-file scoring: input CSV in, same CSV plus a `HADES_Score` column out.

In [6]:
scored = hades.score_csv(DATA, "test_scores.csv")
print(f" scored rows = {len(scored)}")
scored.head()

Wrote 20 scored rows to smoke_test_scores.csv
 scored rows = 20


,SMILES,phase,HADES_Score
0,O=C([O-])COCCN1CCN([C@@H](c2ccccc2)c2ccc(Cl)cc...,4,0.997880
1,COc1c(C)c2c(c(O)c1C/C=C(\C)CCC(=O)[O-])C(=O)OC2,4,0.997274
2,N#C[C@@H]1CCCN1C(=O)CNC12CC3CC(CC(O)(C3)C1)C2,4,0.994455
3,CN1[C@@H]2CC[C@H]1C[C@@H](OC(=O)c1c[nH]c3ccccc...,4,0.997185
4,NC(=O)CN1CCCC1=O,4,0.997188


## 2. Train a model on the bundled sample data

`data_to_test.csv` has 20 molecules with a `phase` column (ChEMBL max phase). The
trainer treats `phase == 4` (approved) as positive and `phase == 0` as negative.

`parameters_test.py` supplies a deliberately small search space so this finishes
quickly — importing it keeps the notebook and that file from drifting apart.

In [ ]:
trainer = HADESTrainer(
    output_dir="test_run",
    smiles_col="SMILES",
    label_col="phase",
    positive_label=4,
    negative_label=0,
    param_spaces=small_param_spaces(),
    overwrite=True,   # re-running the notebook replaces the previous run
)
out_dir = trainer.run(DATA)
print(f"run written to: {out_dir}")

In [ ]:
print("[shape] run artefacts:", sorted(p.name for p in Path(out_dir).iterdir()))

## 3. Score with the model just trained

Point `HADES_score` at the new run directory instead of the published one.

In [ ]:
hades_new = HADES_score(
    models_dir=f"{out_dir}/models",
    x_train_parquet=f"{out_dir}/X_train.parquet",
)
new_batch = hades_new.score(["CCCC", "CCO", "c1ccccc1"])
print(f"[shape] batch scores: {new_batch.shape}")
new_batch

---

test complete. Scores from section 3 come from a 20-molecule toy model and are
**not** meaningful predictions — they only prove the training path runs end to end.